# Notebook 7: Mapa de puntos (lat/lon) — densidad + puntos de importancia

Usa las coordenadas **exactas** de cada incendio (no el polígono de la comuna, que ya se
cubrió en `06_mapa_georreferenciado_comuna.ipynb`) para mostrar dónde ocurren realmente los
eventos y destacar cuáles importan más para el proyecto.

## Criterio de "puntos de importancia" (ligado a la hipótesis del proyecto)

Se marcan dos tipos de evento, no excluyentes:

1. **Catastrófico**: `superficie_ha` en el percentil 99 (el 1% de incendios más grandes).
2. **Estrés hídrico extremo**: el mes del evento fue simultáneamente muy seco
   (`tp_anomaly_mensual` en el percentil 5 más bajo) y muy caluroso (`t2m_anomaly_mensual`
   en el percentil 95 más alto) respecto a la climatología de su propia celda — esta es
   exactamente la señal que el proyecto busca detectar (estrés hídrico anómalo asociado a
   severidad de incendios), no un criterio genérico de "clima extremo".

**Entrada:** el CSV del notebook 03 (`incendios_conaf_era5_2010_2020.csv`) — ya tiene lat/lon,
severidad y anomalías climáticas por evento.


## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Instalar librerías

In [ ]:
!pip install -q folium
print('✓ Librerías instaladas')


## 3. Configuración — EDITA la ruta antes de correr

In [ ]:
import os

# ⚠️ Debe ser la MISMA carpeta base usada en los notebooks anteriores
DRIVE_BASE = '/content/drive/MyDrive/CAMBIAR_A_TU_RUTA'

INPUT_CSV = f'{DRIVE_BASE}/datos_procesados/incendios_conaf_era5_2010_2020.csv'
OUTPUT_DIR = f'{DRIVE_BASE}/datos_procesados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Entrada :', INPUT_CSV)


## 4. Cargar el dataset evento-a-evento (con clima)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(INPUT_CSV)
print(f'✓ {len(df)} eventos cargados')
df[['comuna', 'region', 'latitud', 'longitud', 'superficie_ha',
    't2m_c', 'tp_mm', 't2m_anomaly_mensual', 'tp_anomaly_mensual']].head()


## 5. Calcular el criterio de "puntos de importancia"

Percentiles calculados sobre el propio dataset (60.530 eventos) — se imprimen los umbrales
exactos para que quede documentado qué corte se usó.


In [ ]:
p99_superficie = df['superficie_ha'].quantile(0.99)
p05_tp_anom = df['tp_anomaly_mensual'].quantile(0.05)
p95_t2m_anom = df['t2m_anomaly_mensual'].quantile(0.95)

print(f'Umbral catastrófico (p99 superficie_ha)      : {p99_superficie:.2f} ha')
print(f'Umbral seco extremo (p05 tp_anomaly_mensual) : {p05_tp_anom:.2f} mm')
print(f'Umbral cálido extremo (p95 t2m_anomaly_mensual): {p95_t2m_anom:.2f} °C')

es_catastrofico = df['superficie_ha'] >= p99_superficie
es_estres_extremo = (df['tp_anomaly_mensual'] <= p05_tp_anom) & (df['t2m_anomaly_mensual'] >= p95_t2m_anom)

df['motivo_importancia'] = np.select(
    [es_catastrofico & es_estres_extremo, es_catastrofico, es_estres_extremo],
    ['catastrofico_y_estres_hidrico', 'catastrofico', 'estres_hidrico_extremo'],
    default=None,
)

puntos_importantes = df[df['motivo_importancia'].notna()].copy()
print(f'\n✓ {len(puntos_importantes)} puntos de importancia de {len(df)} eventos totales '
      f'({100 * len(puntos_importantes) / len(df):.1f}%)')
print(puntos_importantes['motivo_importancia'].value_counts())


## 6. Puntos de importancia por región (para saber dónde enfocar en la presentación)

In [ ]:
puntos_importantes.groupby('region')['motivo_importancia'].count().sort_values(ascending=False)


## 7. Mapa: densidad (todos los eventos) + puntos de importancia destacados

- **Heatmap** de los 60.530 eventos, ponderado por `log1p(superficie_ha)` (severidad-ponderada,
  no solo conteo) — capa de fondo, da la vista territorial general.
- **Marcadores individuales** solo para los puntos de importancia (son pocos, se pueden
  inspeccionar uno a uno con popup: comuna, fecha, superficie, clima).
- **Filtro por año** (una capa por año), **por motivo** (solo catastrófico / solo estrés
  hídrico / solo ambos) y **por estación del año** (verano/otoño/invierno/primavera, según el
  mes exacto de cada evento) — independientes entre sí, se pueden combinar activando y
  desactivando casillas del control de capas.


In [ ]:
import folium
from folium.plugins import HeatMap

m = folium.Map(location=[-38.5, -71.5], zoom_start=5, tiles='CartoDB dark_matter')

ESTILO_MOTIVO = {
    'catastrofico':                   {'color': 'red',    'icon': 'fire',              'etiqueta': 'Catastrófico'},
    'estres_hidrico_extremo':         {'color': 'orange', 'icon': 'tint',              'etiqueta': 'Estrés hídrico extremo'},
    'catastrofico_y_estres_hidrico':  {'color': 'purple', 'icon': 'exclamation-triangle', 'etiqueta': 'Ambos a la vez'},
}

def capa_puntos(df_sub, nombre, activa):
    '''Arma una FeatureGroup de marcadores a partir de un subconjunto de puntos_importantes.'''
    fg = folium.FeatureGroup(name=nombre, show=activa)
    for _, row in df_sub.iterrows():
        estilo = ESTILO_MOTIVO[row['motivo_importancia']]
        popup_html = f'''
        <div style="font-family:sans-serif;min-width:200px">
            <h4 style="margin:0">{row['comuna']}, {row['region']}</h4>
            <hr style="margin:4px 0">
            <b>Motivo:</b> {estilo['etiqueta']}<br>
            <b>Fecha:</b> {row['fecha_evento']}<br>
            <b>Superficie:</b> {row['superficie_ha']:.1f} ha<br>
            <b>Temp. día:</b> {row['t2m_c']:.1f} °C (anomalía mensual: {row['t2m_anomaly_mensual']:+.1f})<br>
            <b>Precip. día:</b> {row['tp_mm']:.1f} mm (anomalía mensual: {row['tp_anomaly_mensual']:+.1f})
        </div>
        '''
        folium.Marker(
            location=[row['latitud'], row['longitud']],
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{row['comuna']} — {estilo['etiqueta']}",
            icon=folium.Icon(color=estilo['color'], icon=estilo['icon'], prefix='fa'),
        ).add_to(fg)
    return fg

# Paleta del heatmap deliberadamente azul/cian/amarillo -> NUNCA toca rojo/naranja/morado,
# que son los colores usados para los puntos de importancia (si comparten colores, el ojo no
# distingue "esto es densidad" de "esto es un punto marcado").
heat_data = [
    [row['latitud'], row['longitud'], np.log1p(row['superficie_ha'])]
    for _, row in df.iterrows()
]
fg_heat = folium.FeatureGroup(name='Densidad de incendios (todos los años)', show=False)
HeatMap(
    heat_data, radius=8, blur=10, min_opacity=0.25,
    gradient={'0.2': '#2c3e94', '0.4': '#2f9bda', '0.6': '#39c6c6', '0.8': '#8fe388', '1.0': '#f4e04d'},
).add_to(fg_heat)
fg_heat.add_to(m)

# Capa por defecto: todos los puntos de importancia, todos los años y motivos juntos.
capa_puntos(puntos_importantes, 'Puntos de importancia — todos los años', True).add_to(m)

# Filtro por año — una capa por año, apagadas por defecto (activar para aislar un año).
anios_disponibles = sorted(int(a) for a in puntos_importantes['año'].dropna().unique())
for anio in anios_disponibles:
    sub = puntos_importantes[puntos_importantes['año'] == anio]
    capa_puntos(sub, f'· Año {anio}', False).add_to(m)

# Filtro por motivo — una capa por categoría, apagadas por defecto (activar para aislar una).
NOMBRE_MOTIVO_CAPA = {
    'catastrofico': '· Solo catastrófico',
    'estres_hidrico_extremo': '· Solo estrés hídrico extremo',
    'catastrofico_y_estres_hidrico': '· Solo ambos a la vez',
}
for motivo, nombre_capa in NOMBRE_MOTIVO_CAPA.items():
    sub = puntos_importantes[puntos_importantes['motivo_importancia'] == motivo]
    capa_puntos(sub, nombre_capa, False).add_to(m)

# Filtro por estación del año — cada evento ya tiene 'mes' (fecha exacta), así que esto no
# necesita ningún recálculo previo (a diferencia de los mapas de comuna del notebook 06, que
# están agregados solo por año en Spark). Estaciones calendario reales, no la bandera
# "temporada de incendios" (dic-mar) que se usa en el modelo — son dos cosas distintas.
MES_A_ESTACION = {
    12: 'Verano', 1: 'Verano', 2: 'Verano',
    3: 'Otoño', 4: 'Otoño', 5: 'Otoño',
    6: 'Invierno', 7: 'Invierno', 8: 'Invierno',
    9: 'Primavera', 10: 'Primavera', 11: 'Primavera',
}
puntos_importantes = puntos_importantes.copy()
puntos_importantes['estacion'] = puntos_importantes['mes'].map(MES_A_ESTACION)

for estacion in ['Verano', 'Otoño', 'Invierno', 'Primavera']:
    sub = puntos_importantes[puntos_importantes['estacion'] == estacion]
    capa_puntos(sub, f'· Estación: {estacion}', False).add_to(m)

# Leyenda única: separa "densidad" (barra de gradiente) de "puntos de importancia" (íconos +
# color) y explica cómo usar los filtros — en una sola caja, nada suelto por el mapa.
leyenda = '''
<div style="position:fixed;bottom:20px;left:20px;z-index:1000;
     background:rgba(20,20,20,0.85);color:white;padding:14px 16px;border-radius:10px;
     font-family:sans-serif;font-size:12.5px;border:1px solid #555;max-width:270px;
     box-shadow:0 2px 8px rgba(0,0,0,0.4)">
  <div style="font-weight:600;font-size:13.5px;margin-bottom:8px">Incendios forestales 2010-2019</div>

  <div style="font-weight:600;margin-bottom:4px">Densidad (capa opcional)</div>
  <div style="height:8px;border-radius:4px;margin-bottom:4px;
       background:linear-gradient(to right, #2c3e94, #2f9bda, #39c6c6, #8fe388, #f4e04d)"></div>
  <div style="display:flex;justify-content:space-between;font-size:10.5px;color:#ccc;margin-bottom:10px">
    <span>baja</span><span>alta</span>
  </div>

  <div style="font-weight:600;margin-bottom:4px">Puntos de importancia</div>
  <div>🔥 Catastrófico (top 1% superficie)</div>
  <div>💧 Estrés hídrico extremo (mes seco + cálido)</div>
  <div>⚠️ Ambos a la vez</div>

  <div style="font-weight:600;margin:8px 0 4px 0">Filtros (control de capas →)</div>
  <div style="font-size:11px;color:#ccc">
    Activa "· Año XXXX", "· Solo [motivo]" o "· Estación: [nombre]" para aislar un
    subconjunto — se pueden combinar (ej. desactiva "todos los años" y activa un año puntual
    + una estación).
  </div>
</div>
'''
m.get_root().html.add_child(folium.Element(leyenda))

folium.LayerControl(collapsed=False).add_to(m)

out_mapa = f'{OUTPUT_DIR}/mapa_puntos_importancia.html'
m.save(out_mapa)
print(f'✓ Mapa guardado: {out_mapa} · {len(anios_disponibles)} capas de año + '
      f'{len(NOMBRE_MOTIVO_CAPA)} de motivo + 4 de estación')
m


## ✅ Checklist de validación

- [x] Umbrales del criterio impresos explícitamente (sección 5) — quedan documentados, no son
      un número mágico oculto en el código.
- [x] El heatmap usa TODOS los eventos (vista territorial completa); los marcadores
      individuales son solo el subconjunto de importancia (liviano de renderizar).
- [x] Cada punto de importancia es inspeccionable (popup con comuna, fecha, severidad, clima).

**Para la presentación:** la tabla de la sección 6 (puntos de importancia por región) es un
buen dato para decidir en qué región hacer zoom al mostrar el mapa en vivo.
